# Biomedical intermediate training + reading all the evidence

Two changes to the NLI classifier (`results/nli_classifier_v1.md`, CV macro-F1 0.570):

1. **Intermediate training** on human-labelled biomedical claim verification (HealthVer + SciFact,
   15,589 pairs, `scripts/build_pretrain_data.py`) before fine-tuning on our citations. It is done
   once (`scripts/pretrain_nli.py`), and every fold and seed starts from the result.
2. **1024 tokens instead of 512.** At 512 the classifier sees about 12 of the 20 evidence sentences;
   at 1024, 80% of citations fit entirely. DeBERTa-v3 has no absolute position limit.

- **Phase 0** (about 30 minutes): download the data and do the intermediate training.
- **Phase 1** (about 4 hours): 3-fold cross-validation of (1) at 512 tokens, then (1) + (2) at 1024.
- **Phase 2** (1.5 to 3.5 hours): 3 seeds of the better variant, predicting test. **It runs only if
  that variant beats 0.570.**

This will probably take **two Colab sessions**. Everything is saved to `MyDrive/CiteAudit_runs/v8`;
when Colab disconnects or runs out of GPU time, Run all again (later or the next day) and finished
work is skipped.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v8'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
PRE = f'{RES}/nli_biomed'   # the intermediate-trained model, shared by every run
SIZES = {512: '--max-len 512 --batch 2 --accum 4', 1024: '--max-len 1024 --batch 1 --accum 8 --eval-batch 4 --grad-ckpt'}
def train(name, seed, length, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --model {PRE} --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 {SIZES[length]} --seed {seed} --amp '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Phase 0: intermediate training on HealthVer + SciFact

In [ ]:
if not os.path.exists(f'{PRE}/model.safetensors'):
    run('python scripts/build_pretrain_data.py --out data_train/pretrain/biomed_nli.jsonl')
    run(f'python scripts/pretrain_nli.py --data data_train/pretrain/biomed_nli.jsonl --out {PRE}.part --amp')
    os.rename(f'{PRE}.part', PRE)
else:
    print('intermediate model: already done')

## Phase 1: 3-fold cross-validation over train

In [ ]:
VARIANTS = {'bio512': 512, 'bio1024': 1024}
BASE = [0.676, 0.570, 0.783, 0.513, 0.415]   # NLI without intermediate training, 512 tokens (nli_classifier_v1.md)
cv = {}
for v, length in VARIANTS.items():
    for k in range(3):
        train(f'cv_{v}_f{k}', 0, length, f'--folds 3 --fold {k}')
    cv[v] = scores(sum((read(f'{RES}/cv_{v}_f{k}/oof.jsonl') for k in range(3)), []))
    print(f'{v}: ' + ' '.join(f'{c} {x:.3f}' for c, x in zip(COLS, cv[v])), flush=True)
print(f'\n{"cross-validated, 2141 citations":34s} ' + ' '.join(f'{c:>9s}' for c in COLS))
print(f'{"NLI, 512 (current best)":34s} ' + ' '.join(f'{x:9.3f}' for x in BASE))
for v in VARIANTS:
    print(f'{"+ biomedical, " + str(VARIANTS[v]) + " tokens":34s} ' + ' '.join(f'{x:9.3f}' for x in cv[v]))
BEST = max(VARIANTS, key=lambda v: cv[v][1])
GO = cv[BEST][1] > BASE[1]
json.dump({'cv': cv, 'nli_cv': BASE, 'best': BEST, 'phase2': GO}, open(f'{RES}/cv_summary.json', 'w'), indent=2)
print(f'\nPhase 2 will run with {BEST}.' if GO else '\nNeither variant beats the current best; Phase 2 is skipped.')

## Phase 2: 3 seeds of the better variant, predicting test (only if Phase 1 won)

In [ ]:
if GO:
    for seed in (0, 1, 2):
        train(f'final_{BEST}_s{seed}', seed, VARIANTS[BEST], f'--predict test={T}/ours_20r_test')
    runs = [scores(read(f'{RES}/final_{BEST}_s{s}/ours_20r_test.jsonl')) for s in (0, 1, 2)]
    print('test, mean ± sd over 3 seeds')
    for name, xs in zip(COLS, zip(*runs)):
        print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
    print('current best on test: accuracy 0.668 ± 0.015, macro-F1 0.556 ± 0.015 (single seed); '
          '0.680 / 0.575 with 3 seeds averaged')
    run(f'python scripts/tune_bias.py --oof {RES}/cv_{BEST}_f0/oof.jsonl {RES}/cv_{BEST}_f1/oof.jsonl '
        f'{RES}/cv_{BEST}_f2/oof.jsonl --apply ' + ' '.join(f'{RES}/final_{BEST}_s{s}/ours_20r_test.jsonl' for s in (0, 1, 2))
        + f' --out {RES}/bias.json')

## Download

Zips `MyDrive/CiteAudit_runs/v8` (predictions and logs, without the model weights). Put `v8.zip` in
the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v8.zip v8 -x '*.part/*' -x 'v8/nli_biomed/*'
from google.colab import files
files.download('/content/v8.zip')